### Building a scalable ML pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Pipeline should contain all methods except splitting.
# Splitting should be done outside the pipeline.

import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import mean_absolute_error, make_scorer, classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as imbpipeline
from imblearn.over_sampling import SMOTENC

In [ ]:
# The goal of this ML model is to check how accurate it can be in predicting each class
# That is, we reduce the error in prediction as much as possible

In [ ]:
# Import the dataset
import pandas as pd

train_df = pd.read_csv(r'/content/drive/MyDrive/financial-inclusion-in-africa20250311-22142-nbnoiv/Train.csv')
test_df = pd.read_csv(r'/content/drive/MyDrive/financial-inclusion-in-africa20250311-22142-nbnoiv/Test.csv')

In [ ]:
train_df.head()

,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3,24,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5,70,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5,26,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5,34,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8,26,Male,Child,Single/Never Married,Primary education,Informally employed


In [ ]:
# Binary Columns
train_df['bank_account'] = train_df['bank_account'].map({'Yes': 1, 'No': 0})
train_df['location_type'] = train_df['location_type'].map({'Rural': 1, 'Urban': 0})
train_df['cellphone_access'] = train_df['cellphone_access'].map({'Yes': 1, 'No': 0})
train_df['gender_of_respondent'] = train_df['gender_of_respondent'].map({'Female': 1, 'Male': 0})

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   country                 23524 non-null  object
 1   year                    23524 non-null  int64 
 2   uniqueid                23524 non-null  object
 3   bank_account            23524 non-null  int64 
 4   location_type           23524 non-null  int64 
 5   cellphone_access        23524 non-null  int64 
 6   household_size          23524 non-null  int64 
 7   age_of_respondent       23524 non-null  int64 
 8   gender_of_respondent    23524 non-null  int64 
 9   relationship_with_head  23524 non-null  object
 10  marital_status          23524 non-null  object
 11  education_level         23524 non-null  object
 12  job_type                23524 non-null  object
dtypes: int64(7), object(6)
memory usage: 2.3+ MB


In [ ]:
train_df1 = train_df.drop(columns='uniqueid')

In [ ]:
train_df1.head()

,country,year,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,1,1,1,3,24,1,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,0,1,0,5,70,1,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,1,0,1,5,26,0,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,0,1,1,5,34,1,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,0,0,0,8,26,0,Child,Single/Never Married,Primary education,Informally employed


In [ ]:
# Split data into X and y
X = train_df1.drop(columns='bank_account')
y = train_df1['bank_account']

In [ ]:
X.head(1)

,country,year,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,1,1,3,24,1,Spouse,Married/Living together,Secondary education,Self employed


In [ ]:
y.head(1)

,bank_account
0,1


In [ ]:
# Now, we split the dataset
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# Divide X_train into different categories based on datatype
numeric_cols = ['age_of_respondent', 'household_size']
binary_cols = ['cellphone_access', 'location_type', 'gender_of_respondent']
nominal_cols = ['country', 'relationship_with_head', 'marital_status', 'education_level', 'job_type']
cat_idx = [X_train.columns.get_loc(col) for col in nominal_cols] # You can also just pass nominal_cols instead of the index too
# This is even much safer because if you ever reorder your columns, the code might break

In [ ]:
preprocessor = ColumnTransformer([
    ('num', MinMaxScaler(feature_range=(0, 1)), numeric_cols),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), nominal_cols),
     ], remainder='passthrough')

# remainder='passthrough' allows my binary columns and any other column not in the numeric or categorical frame go through

In [ ]:
# Create the model as well
model = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)

In [ ]:
# Create the SMOTENC re-sampling method using cat_idx
smotenc = SMOTENC(categorical_features=nominal_cols, random_state=42) # I will just pass nominal_cols instead

In [ ]:
# Now, that we have all of the methods prepared, let's pass it into a "step" variable
steps = [
     ('smote', smotenc),
     ('prep', preprocessor),
     ('model', model)
     ]

In [ ]:
pipe = imbpipeline(steps)

In [ ]:
# Only do hyperparameter tuning after creating pipeline so we can pass it in as an estimator
# Define the hyperparameter grid on the pipeline
# Format must be StepName__parameterName
from scipy.stats import randint, loguniform
param_grid = {
    'model__n_estimators': randint(50, 300),
    'model__max_depth': randint(3, 20),
    'model__learning_rate': loguniform(0.001, 0.3)
}

# Then we wrap the pipeline inside of the hyperparameter grid
random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    random_state=42,
    n_jobs=-1,
    scoring='accuracy' # Since this is what we are trying to improve
)

In [ ]:
# 1. Start the search (This does the fitting!)
# Why? This is because when we run .fit(...) on random_search, it activates the parameter tuning which in turn
# activates the pipeline which activates the preprocessor, resampler and model
# Reason for this is that we fit on the outermost layer. If i run pipe.fit(...) here, it will ignore the parameter
# tuning and jump straight to the pipeline thereby running my model on the default settings

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('smote',
                                              SMOTENC(categorical_features=['country',
                                                                            'relationship_with_head',
                                                                            'marital_status',
                                                                            'education_level',
                                                                            'job_type'],
                                                      random_state=42)),
                                             ('prep',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('num',
                                                                               MinMaxScaler(),
                                                                               ['age_of_respondent',
                                                                                'household_size']),
                                                                              ('cat',
                                                                               OneHotEncoder(handle_unknown...
                   param_distributions={'model__learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7de4f5b05cd0>,
                                        'model__max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7de4f593a850>,
                                        'model__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7de4f5938950>},
                   random_state=42, scoring='accuracy')

In [ ]:
# 2. Extract the best fitted pipeline
best_pipeline = random_search.best_estimator_

In [ ]:
# 3. Use this to see the parameters that won
best_pipeline.named_steps['model']

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None,
              learning_rate=np.float64(0.13983740016490973), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=153, n_jobs=-1,
              num_parallel_tree=None, ...)

In [ ]:
# Let's check the score of the base model and tuned model
from sklearn.metrics import accuracy_score

# 1. Baseline Model (Default XGBoost settings)
pipe.fit(X_train, y_train)
base_preds = pipe.predict(X_val)
base_acc = accuracy_score(y_val, base_preds)
print(f"Baseline Accuracy: {base_acc:.4f}")

# 2. Tuned Model (Winning hyperparameters from Random Search)
tuned_preds = best_pipeline.predict(X_val)
tuned_acc = accuracy_score(y_val, tuned_preds)
print(f"Tuned Accuracy:    {tuned_acc:.4f}")

Baseline Accuracy: 0.8742
Tuned Accuracy:    0.8765


In [ ]:
# Get probabilities once
val_proba = best_pipeline.predict_proba(X_val)[:, 1]

# Try a range of thresholds
for t in np.arange(0.1, 1.0, 0.05):
    preds = (val_proba >= t).astype(int)
    mae = mean_absolute_error(y_val, preds)
    acc = 1 - mae
    print(f"Threshold: {t:.2f} | MAE: {mae:.4f} | Accuracy: {acc:.4f}")

Threshold: 0.10 | MAE: 0.3147 | Accuracy: 0.6853
Threshold: 0.15 | MAE: 0.2460 | Accuracy: 0.7540
Threshold: 0.20 | MAE: 0.2056 | Accuracy: 0.7944
Threshold: 0.25 | MAE: 0.1774 | Accuracy: 0.8226
Threshold: 0.30 | MAE: 0.1598 | Accuracy: 0.8402
Threshold: 0.35 | MAE: 0.1462 | Accuracy: 0.8538
Threshold: 0.40 | MAE: 0.1353 | Accuracy: 0.8647
Threshold: 0.45 | MAE: 0.1271 | Accuracy: 0.8729
Threshold: 0.50 | MAE: 0.1235 | Accuracy: 0.8765
Threshold: 0.55 | MAE: 0.1199 | Accuracy: 0.8801
Threshold: 0.60 | MAE: 0.1167 | Accuracy: 0.8833
Threshold: 0.65 | MAE: 0.1159 | Accuracy: 0.8841
Threshold: 0.70 | MAE: 0.1153 | Accuracy: 0.8847
Threshold: 0.75 | MAE: 0.1172 | Accuracy: 0.8828
Threshold: 0.80 | MAE: 0.1177 | Accuracy: 0.8823
Threshold: 0.85 | MAE: 0.1211 | Accuracy: 0.8789
Threshold: 0.90 | MAE: 0.1279 | Accuracy: 0.8721
Threshold: 0.95 | MAE: 0.1366 | Accuracy: 0.8634


In [ ]:
# Get probabilities once
val_proba = pipe.predict_proba(X_val)[:, 1]

# Try a range of thresholds
for t in np.arange(0.1, 1.0, 0.05):
    preds = (val_proba >= t).astype(int)
    mae = mean_absolute_error(y_val, preds)
    acc = 1 - mae
    print(f"Threshold: {t:.2f} | MAE: {mae:.4f} | Accuracy: {acc:.4f}")

Threshold: 0.10 | MAE: 0.3080 | Accuracy: 0.6920
Threshold: 0.15 | MAE: 0.2393 | Accuracy: 0.7607
Threshold: 0.20 | MAE: 0.1995 | Accuracy: 0.8005
Threshold: 0.25 | MAE: 0.1782 | Accuracy: 0.8218
Threshold: 0.30 | MAE: 0.1602 | Accuracy: 0.8398
Threshold: 0.35 | MAE: 0.1469 | Accuracy: 0.8531
Threshold: 0.40 | MAE: 0.1370 | Accuracy: 0.8630
Threshold: 0.45 | MAE: 0.1316 | Accuracy: 0.8684
Threshold: 0.50 | MAE: 0.1258 | Accuracy: 0.8742
Threshold: 0.55 | MAE: 0.1216 | Accuracy: 0.8784
Threshold: 0.60 | MAE: 0.1194 | Accuracy: 0.8806
Threshold: 0.65 | MAE: 0.1194 | Accuracy: 0.8806
Threshold: 0.70 | MAE: 0.1199 | Accuracy: 0.8801
Threshold: 0.75 | MAE: 0.1194 | Accuracy: 0.8806
Threshold: 0.80 | MAE: 0.1194 | Accuracy: 0.8806
Threshold: 0.85 | MAE: 0.1226 | Accuracy: 0.8774
Threshold: 0.90 | MAE: 0.1281 | Accuracy: 0.8719
Threshold: 0.95 | MAE: 0.1357 | Accuracy: 0.8643


In [ ]:
# 1. Train/Test Split

# 2. Split your columns into numerical, binary and categorical column. Also create cat_idx when resampling

# 3. Build the Base Pipeline

# 4. Define the Hyperparameter Grid
      # Format must be: 'stepName__parameterName'
'''
param_distributions = {
    'sampler__k_neighbors':,                      # Tuning SMOTENC parameters
    'classifier__n_estimators':,              # Tuning Model parameters
    'classifier__max_depth': [None, 5, 10, 20],
    'classifier__min_samples_split': [2, 5, 10]
}
'''
# 5. Wrap the Pipeline inside RandomizedSearchCV
# n_iter=5 means it will randomly pick and test 5 parameter combinations
''''
random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=5,
    cv=3,
    random_state=42,
    n_jobs=-1,
    scoring='accuracy' # Best metric for imbalanced data!
)
'''

# 6. Run the Search
# This uses cross-validation internally. For every single fold:
#   - It splits training/validation data
#   - Applies SMOTENC ONLY to the training fold
#   - Trains the model with a set of hyperparameters
#   - Tests on the validation fold (skipping SMOTENC, keeping hyperparameters)
'''
random_search.fit(X_train, y_train)
'''
# 7. Evaluate on the final Test Set
# Under the hood, this uses the absolute best hyperparameters found, but skips SMOTENC!
'''
final_score = random_search.score(X_test, y_test)
'''

'''
print(f"Best Found Parameters: {random_search.best_params_}")
print(f"Final Test Set F1-Score: {final_score:.4f}")
'''

'\nprint(f"Best Found Parameters: {random_search.best_params_}")\nprint(f"Final Test Set F1-Score: {final_score:.4f}")\n'

In [ ]:
test_df.head(5)

,country,year,uniqueid,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_6056,Urban,Yes,3,30,Male,Head of Household,Married/Living together,Secondary education,Formally employed Government
1,Kenya,2018,uniqueid_6060,Urban,Yes,7,51,Male,Head of Household,Married/Living together,Vocational/Specialised training,Formally employed Private
2,Kenya,2018,uniqueid_6065,Rural,No,3,77,Female,Parent,Married/Living together,No formal education,Remittance Dependent
3,Kenya,2018,uniqueid_6072,Rural,No,6,39,Female,Head of Household,Married/Living together,Primary education,Remittance Dependent
4,Kenya,2018,uniqueid_6073,Urban,No,3,16,Male,Child,Single/Never Married,Secondary education,Remittance Dependent


In [ ]:
# Fit the entire data into the model now
# then we can pass into the random_search as random_search.fit(X, y)
# To do the test prediction,
# drop uniqueid
# map location_type, cellphone_access, gender_of_respondent
# then pass it into random_search.predict(test_df)

In [ ]:
test_df.columns

Index(['country', 'year', 'uniqueid', 'location_type', 'cellphone_access',
       'household_size', 'age_of_respondent', 'gender_of_respondent',
       'relationship_with_head', 'marital_status', 'education_level',
       'job_type'],
      dtype='object')

In [ ]:
# 1. FINAL REFIT
print("Refitting best pipeline on all data...")
best_pipeline.fit(X, y)

# 2. CLEAN TEST_DF
# Drop the columns that the model shouldn't see
test_df_clean = test_df.drop(columns=['uniqueid'], errors='ignore')

# Map the binary columns (using replace to avoid NaNs)
test_df_clean['location_type'] = test_df_clean['location_type'].replace({'Rural': 1, 'Urban': 0})
test_df_clean['cellphone_access'] = test_df_clean['cellphone_access'].replace({'Yes': 1, 'No': 0})
test_df_clean['gender_of_respondent'] = test_df_clean['gender_of_respondent'].replace({'Female': 1, 'Male': 0})

# 3. PREDICT
test_proba = best_pipeline.predict_proba(test_df_clean)[:, 1]
final_test_preds = (test_proba >= 0.70).astype(int)

# 4. FORMAT
# We use the ORIGINAL test_df for the IDs, because we didn't drop them there!

# 1. Create the DataFrame
submission = pd.DataFrame({
    'uniqueid': test_df['uniqueid'] + " x " + test_df['country'],
    'bank_account_pred': final_test_preds, # This is your 1s and 0s
})

# 2. Add the mapped Yes/No column based on the raw column
submission['bank_account'] = submission['bank_account_pred'].map({1: 'Yes', 0: 'No'})


# 5. SAVE
submission.to_csv('zindi_submission.csv', index=False)

Refitting best pipeline on all data...


In [ ]:
submission.head(5)

,uniqueid,bank_account_pred,bank_account
0,uniqueid_6056 x Kenya,1,Yes
1,uniqueid_6060 x Kenya,1,Yes
2,uniqueid_6065 x Kenya,0,No
3,uniqueid_6072 x Kenya,0,No
4,uniqueid_6073 x Kenya,0,No


In [ ]:
final_test_preds

array([1, 1, 0, ..., 0, 0, 0])

In [ ]:
test_proba

array([0.8564726 , 0.90153044, 0.15438253, ..., 0.4741401 , 0.02645191,
       0.03040914], dtype=float32)